# Learning Objectives

- Build an LLM assistant for document-based Q&A using retrieval-augmented generation.

# Setup

In [1]:


import time
import chromadb

from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_core.documents import Document

from langchain_chroma import Chroma

from datetime import datetime

#from google.colab import userdata


c:\Users\Aditya Sodani\Desktop\AG0855_2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from dotenv import load_dotenv
load_dotenv()
import os
os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')

Below is the api key, endpoint, api version, deployment name for the chat model

In [3]:
from groq import Groq

client = Groq()

In [4]:
model_name = 'openai/gpt-oss-120b'

 Instantiating the embedding model

In [5]:
### Before running the command ensure the numpy version is 2.3.4. If not then upgrade numpy as per previous steps in notebook.
### Then do not restart the notebook. If you restart then you will loose all loaded variables. Just click Cancel instad of Restart

from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

C:\Users\Aditya Sodani\AppData\Local\Temp\ipykernel_2628\1865801256.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")


# Download Raw Data


In [9]:
import zipfile
 
zip_path = r"C:\Users\Aditya Sodani\Desktop\AG0855_2\tesla-annual-reports.zip"
extract_to = r"C:\Users\Aditya Sodani\Desktop\AG0855_2"
 
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)
 
print("Unzipped successfully!")

Unzipped successfully!


# Chunking

In [6]:
from pathlib import Path
pdf_folder_location = Path("tesla-annual-reports").resolve()
print("Loading PDFs from:", pdf_folder_location)

Loading PDFs from: C:\Users\Aditya Sodani\Desktop\AG0855_2\tesla-annual-reports


In [7]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
pdf_loader = PyPDFDirectoryLoader(str(pdf_folder_location))

In [8]:
type(pdf_loader)

langchain_community.document_loaders.pdf.PyPDFDirectoryLoader

In [9]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap=16
)

In [10]:
tesla_10k_chunks = pdf_loader.load_and_split(text_splitter)

In [ ]:
# from groq import Groq

# # Replace with your Groq API Key
# API_KEY = "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"

# client = Groq(api_key=API_KEY)

# try:
#     response = client.chat.completions.create(
#         model="llama-3.3-70b-versatile",
#         messages=[
#             {
#                 "role": "user",
#                 "content": "Say hello in one sentence."
#             }
#         ],
#         temperature=0
#     )

#     print("✅ API Key is working!")
#     print("\nResponse:")
#     print(response.choices[0].message.content)

# except Exception as e:
#     print("❌ Error:")
#     print(e)

✅ API Key is working!

Response:
Hello, it's nice to meet you and I'm here to help with any questions or topics you'd like to discuss.


In [12]:
len(tesla_10k_chunks)

3337

In [12]:
print(tesla_10k_chunks[0])

page_content='UNITED	STATES
SECURITIES	AND	EXCHANGE	COMMISSION
Washington,	D.C.	20549
	
FORM	
10-K/A
(Amendment	No.	1)
	
(Mark	One)
☒
ANNUAL	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES	EXCHANGE	ACT	OF	1934
For	the	fiscal	year	ended	
December	31,	
2021
	
OR
☐
TRANSITION	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES	EXCHANGE	ACT	OF	1934
For	the	transition	period	from	
																				
	to	
																					
Commission	File	Number:	
001-34756
	
Tesla,	Inc.
(Exact	name	of	registrant	as	specified	in	its	charter)
	
	
Delaware
	
91-2197729
(State	or	other	jurisdiction	of
incorporation	or	organization)
	
(I.R.S.	Employer
Identification	No.)
	
	
	
1	Tesla	Road
Austin
,	
Texas
	
78725
(Address	of	principal	executive	offices)
	
(Zip	Code)
(
512
)	
516-8177
(Registrant’s	telephone	number,	including	area	code)
Securities	registered	pursuant	to	Section	12(b)	of	the	Act:
	
Title	of	each	class
Trading	Symbol(s)
Name	of	each	exchange	on	which	registered
Common	stock
TSLA

In [13]:
print(tesla_10k_chunks[0].page_content)

UNITED	STATES
SECURITIES	AND	EXCHANGE	COMMISSION
Washington,	D.C.	20549
	
FORM	
10-K/A
(Amendment	No.	1)
	
(Mark	One)
☒
ANNUAL	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES	EXCHANGE	ACT	OF	1934
For	the	fiscal	year	ended	
December	31,	
2021
	
OR
☐
TRANSITION	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES	EXCHANGE	ACT	OF	1934
For	the	transition	period	from	
																				
	to	
																					
Commission	File	Number:	
001-34756
	
Tesla,	Inc.
(Exact	name	of	registrant	as	specified	in	its	charter)
	
	
Delaware
	
91-2197729
(State	or	other	jurisdiction	of
incorporation	or	organization)
	
(I.R.S.	Employer
Identification	No.)
	
	
	
1	Tesla	Road
Austin
,	
Texas
	
78725
(Address	of	principal	executive	offices)
	
(Zip	Code)
(
512
)	
516-8177
(Registrant’s	telephone	number,	including	area	code)
Securities	registered	pursuant	to	Section	12(b)	of	the	Act:
	
Title	of	each	class
Trading	Symbol(s)
Name	of	each	exchange	on	which	registered
Common	stock
TSLA
The	Nasdaq	Gl

# Database Creation

In [13]:
chromadb_client = chromadb.PersistentClient(
    path="./tesla_db"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [16]:
chromadb_client.heartbeat()

1780330781988835400

In [15]:
chromadb_client

In [14]:
tesla_10k_collection = 'tesla-10k-2019-to-2023'

In [15]:
vectorstore = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [18]:
chromadb_client.count_collections()

2

In [56]:
i = 0 # Initialize the starting index for the chunks

while i < len(tesla_10k_chunks): # Iterate while the index is less than the total number of chunks
    vectorstore.add_documents( # Add documents to the vector store in batches of 500
        documents=tesla_10k_chunks[i:i+500], # Get the current batch of 500 chunks
        ids=["text_" + str(i) for i in range(i, i+500)] # Assign unique IDs to each chunk in the batch
    )

    i += 500 # Increment the index by 500 to move to the next batch
    time.sleep(5) # Pause for 30 seconds to avoid rate limiting issues with the vector store

In [19]:
vectorstore_persisted = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [58]:
retriever = vectorstore_persisted.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

In [59]:
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001EC19107E50>, search_kwargs={'k': 5})

# RAG Q&A

## Prompt Design

The RAG system message should clearly communicate to the LLM that the input will include a user query along with the necessary context information to address that query. Additionally, the response should rely solely on the context information provided.

In [60]:
qna_system_message = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

User queries will be delimited by: <Question> and </Question>.

Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

If the answer is not found in the context, respond "I don't know".
"""

In [61]:
qna_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""

## Retrieving relevant documents

In [62]:
user_query = "What was the automotive revenue in 2021?"

In [63]:
relevant_document_chunks = retriever.invoke(user_query)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [64]:
len(relevant_document_chunks)

5

We can inspect the first document like so:

In [ ]:
for document in relevant_document_chunks:
    print(document)
    break


In [ ]:
i = 0
for document in relevant_document_chunks:
    print(f"-------Chunk {i}-------")
    print(document.page_content.replace("\t", " "))

    i += 1

## Composing the response

To compose the response to user queries, we assemble the prompt that uses the system message defined above and the dynamically retrieved context for the user query.

In [67]:
user_query = "What was the automotive revenue in 2021?"

In [69]:
#Irrelevent Query
user_query="what is the total revenue of company in 2020?"

In [71]:
#Irrelevent Query
user_query="what is CJP?"

In [72]:

model_name = 'openai/gpt-oss-120b'

relevant_document_chunks = retriever.invoke(user_query)
context_list = [d.page_content for d in relevant_document_chunks]
context_for_query = "\n---\n".join(context_list)

prompt = [
    {'role': 'system', 'content': qna_system_message},
    {'role': 'user', 'content': qna_user_message_template.format(
         context=context_for_query,
         question=user_query
        )
    }
]


try:
    response = client.chat.completions.create(
        model=model_name,
        messages=prompt,
        temperature=0
    )

    prediction = response.choices[0].message.content.strip()
except Exception as e:
    prediction = f'Sorry, I encountered the following error: \n {e}'

print(prediction)

I don't know.


# Query Expansion Technique

In [108]:
retriever = vectorstore_persisted.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

In [109]:
query_expansion_system_message = """
You are an financial domain expert assisting in answering questions related to 10-k reports.
Perform query expansion on the question below. If there are multiple common ways of phrasing a user question \
or common synonyms for key words in the question, make sure to return multiple versions \
of the query with the different phrasings.

If there are acronyms or words you are not familiar with, do not try to rephrase them.

Return at least 3 versions of the question as a list.
Generate only a list of questions, each question in a new line.
Do not number the list of questions or use bullet points.
Do not mention anything before or after the list.
"""

user_message_template="""
<Question>
{question}
</Question>
"""

In [110]:
user_input = "Any specific fines levied on the company in 2022?"

In [111]:
model_name='openai/gpt-oss-120b'

In [112]:
prompt=[
    {'role':'system', 'content':query_expansion_system_message},
    {'role':'user',   'content': user_message_template.format(
        question=user_input
    )}
]

In [113]:
prompt

[{'role': 'system',
  'content': '\nYou are an financial domain expert assisting in answering questions related to 10-k reports.\nPerform query expansion on the question below. If there are multiple common ways of phrasing a user question or common synonyms for key words in the question, make sure to return multiple versions of the query with the different phrasings.\n\nIf there are acronyms or words you are not familiar with, do not try to rephrase them.\n\nReturn at least 3 versions of the question as a list.\nGenerate only a list of questions, each question in a new line.\nDo not number the list of questions or use bullet points.\nDo not mention anything before or after the list.\n'},
 {'role': 'user',
  'content': '\n<Question>\nAny specific fines levied on the company in 2022?\n</Question>\n'}]

In [114]:
query_expansions = client.chat.completions.create(model=model_name,
                                                  messages=prompt,
                                                  temperature=0)

In [115]:
type(query_expansions.choices[0].message.content)

str

In [116]:
print(query_expansions.choices[0].message.content)

Any specific fines levied on the company in 2022?  
Were any fines imposed on the company during 2022?  
What penalties or fines, if any, were assessed against the company in 2022?


In [117]:
print(query_expansions)

ChatCompletion(id='chatcmpl-43641593-de36-4222-bfbb-910caa4dc7f9', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Any specific fines levied on the company in 2022?  \nWere any fines imposed on the company during 2022?  \nWhat penalties or fines, if any, were assessed against the company in 2022?', role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning='The user wants query expansion: produce multiple versions of the question. Provide at least 3 versions, each on new line, no numbering or bullets. Should not add anything else. So we need to rephrase: "Any specific fines levied on the company in 2022?" synonyms: "Were there any fines imposed on the company in 2022?" "Did the company receive any penalties or fines in 2022?" "What fines, if any, were assessed against the company during 2022?" Also maybe "Were any regulatory fines assessed against the firm in 2022?" Provide at least 3. Provide list, each l

In [118]:
print(query_expansions.choices[-1].message.reasoning)

The user wants query expansion: produce multiple versions of the question. Provide at least 3 versions, each on new line, no numbering or bullets. Should not add anything else. So we need to rephrase: "Any specific fines levied on the company in 2022?" synonyms: "Were there any fines imposed on the company in 2022?" "Did the company receive any penalties or fines in 2022?" "What fines, if any, were assessed against the company during 2022?" Also maybe "Were any regulatory fines assessed against the firm in 2022?" Provide at least 3. Provide list, each line separate. No bullet points. So just lines.


In [119]:
query_expansions_list = query_expansions.choices[0].message.content.strip().split("\n")

In [120]:
len(query_expansions_list)

3

In [121]:
query_expansions_list

['Any specific fines levied on the company in 2022?  ',
 'Were any fines imposed on the company during 2022?  ',
 'What penalties or fines, if any, were assessed against the company in 2022?']

In [122]:
expanded_context_list = []

In [123]:
for query in query_expansions_list:
    expanded_context_list.extend([d.page_content for d in retriever.invoke(query)])

In [124]:
len(expanded_context_list)

9

In [125]:
expanded_context_list

['222\n\t\nDecreases\tin\tbalances\trelated\tto\texpiration\tof\tthe\tstatute\tof\tlimitations\n(\n7\n)\nDecember\t31,\t2022\n870\n\t\nIncreases\tin\tbalances\trelated\tto\tprior\tyear\ttax\tpositions\n59\n\t\nDecreases\trelated\tto\tsettlement\twith\ttax\tauthorities\n(\n6\n)\nIncreases\tin\tbalances\trelated\tto\tcurrent\tyear\ttax\tpositions\n255\n\t\nDecreases\tin\tbalances\trelated\tto\texpiration\tof\tthe\tstatute\tof\tlimitations\n(\n4\n)\nDecember\t31,\t2023\n$\n1,174\n\t\nWe\tinclude\tinterest\tand\tpenalties\trelated\tto\tunrecognized\ttax\tbenefits\tin\tincome\ttax\texpense.\tWe\trecognized\tnet\tinterest\tand\tpenalties\trelated\tto\nunrecognized\ttax\tbenefits\tin\tprovision\tfor\tincome\ttaxes\tline\tof\tour\tconsolidated\tstatements\tof\toperations\tof\t$\n17\n\tmillion,\t$\n27\n\tmillion\tand\t$\n4\n\tmillion\tfor\tthe\nyears\tended\tDecember\t31,\t2023,\t2022\tand\t2021,\trespectively.\tAs\tof\tDecember\t31,\t2023,\tand\t2022,\twe\thave\taccrued\t$\n47\n\tmillion\tand\

In [126]:
final_context_documents = set(expanded_context_list)

In [127]:
len(final_context_documents)

4

In [128]:
qna_system_message = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

User queries will be delimited by: <Question> and </Question>.

Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

If the answer is not found in the context, respond "I don't know".
"""

In [129]:
qna_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""

In [130]:
model_name = 'openai/gpt-oss-120b'

prompt = [
    {'role': 'system', 'content': qna_system_message},
    {'role': 'user', 'content': qna_user_message_template.format(
         context=final_context_documents,
         question=user_input
        )
    }
]

try:
    response = client.chat.completions.create(
        model=model_name,
        messages=prompt,
        temperature=0
    )

    prediction = response.choices[0].message.content.strip()
except Exception as e:
    prediction = f'Sorry, I encountered the following error: \n {e}'

print(prediction)


Yes. For 2022 the company recognized net interest and penalties of $27 million related to unrecognized tax benefits, and as of December 31 2022 it had accrued an additional $31 million of interest and penalties on those unrecognized tax benefits.


# Hypothetical Questions Technique

In [16]:
model_name='llama-3.1-8b-instant'

In [24]:
hypothetical_questions_system_message = """
Generate a list of exactly 3 hypothetical questions that the document presented in the input could be used to answer.
Generate only a list of questions, each question in a new line.
Do not number the questions or use bullet points.
Do not mention anything before or after the list.
"""

user_message_template = """
<Document>
{chunk}
</Document>
"""

In [25]:
hypothetical_questions = []

In [26]:
# def generate_questions(chunk_text):

#     prompt = [
#         {
#             "role": "system",
#             "content": "hypothetical_questions_system_message"
#         },
#         {
#             "role": "user",
#             "content": user_message_template.format(
#                 chunk=chunk_text
#             )
#         }
#     ]

#     response = client.chat.completions.create(
#         model=model_name,
#         messages=prompt,
#         temperature=0
#     )

#     questions = response.choices[0].message.content.strip().split("\n")

#     return questions

In [27]:
# for i, chunk in enumerate(tesla_10k_chunks):

#     questions = generate_questions(chunk.page_content)

#     for q in questions:

#         hypothetical_questions.append(
#             Document(
#                 page_content=q,
#                 metadata={
#                     "chunk_id": i,
#                     "source_text": chunk.page_content
#                 }
#             )
#         )

In [ ]:
BATCH_SIZE = 20

chunk_batches = []

for i in range(0, len(tesla_10k_chunks), BATCH_SIZE):

    batch = tesla_10k_chunks[i:i+BATCH_SIZE]

    batch_text = "\n\n".join(
        [doc.page_content for doc in batch]
    )

    chunk_batches.append(batch_text)

print(f"Total batches: {len(chunk_batches)}")

Total batches: 17


In [32]:
def generate_hypothetical_questions(batch_text):

    response = client.chat.completions.create(
        model=model_name,
        temperature=0,
        messages=[
            {
                "role":"system",
                "content":"hypothetical_questions_system_message"
            },
            {
                "role":"user",
                "content": user_message_template.format(
                    chunk=batch_text
                )
            }
        ]
    )

    return response.choices[0].message.content

In [33]:
batch_questions = []

for idx, batch_text in enumerate(chunk_batches):

    questions = generate_hypothetical_questions(batch_text)

    batch_questions.append({
        "batch_id": idx,
        "questions": questions,
        "context": batch_text
    })

    print(f"Processed Batch {idx+1}")

APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `llama-3.1-8b-instant` in organization `org_01kt0xkpjhedcv9s3a5ffzsd8j` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Requested 72236, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
hypothetical_questions

[Document(metadata={'parent_chunk_id': 'text_0', 'parent_collection': 'tesla-10k-2019-to-2023'}, page_content=''),
 Document(metadata={'parent_chunk_id': 'text_1', 'parent_collection': 'tesla-10k-2019-to-2023'}, page_content=''),
 Document(metadata={'parent_chunk_id': 'text_2', 'parent_collection': 'tesla-10k-2019-to-2023'}, page_content=''),
 Document(metadata={'parent_chunk_id': 'text_3', 'parent_collection': 'tesla-10k-2019-to-2023'}, page_content=''),
 Document(metadata={'parent_chunk_id': 'text_4', 'parent_collection': 'tesla-10k-2019-to-2023'}, page_content=''),
 Document(metadata={'parent_chunk_id': 'text_5', 'parent_collection': 'tesla-10k-2019-to-2023'}, page_content=''),
 Document(metadata={'parent_chunk_id': 'text_6', 'parent_collection': 'tesla-10k-2019-to-2023'}, page_content=''),
 Document(metadata={'parent_chunk_id': 'text_7', 'parent_collection': 'tesla-10k-2019-to-2023'}, page_content=''),
 Document(metadata={'parent_chunk_id': 'text_8', 'parent_collection': 'tesla-10k

In [26]:
hypothetical_questions_vectorstore = Chroma(
    collection_name="hypothetical_questions",
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [29]:
chromadb_client.list_collections()

['tesla-10k-2019-to-2023', 'hypothetical_questions']

In [27]:
hypothetical_questions_vectorstore.add_documents(
    documents=hypothetical_questions
)

['f1b91ed5-b7c2-4da9-aea2-8567b3ed9feb',
 'd5521554-4133-450d-b866-f975f9d6156d',
 'b3803e4c-cf5d-450b-8f09-1957bdb8b6b2',
 '065cfa2e-d2f7-4fd3-9cf8-58ef2fdbbf4d',
 '86ce54a5-11de-4198-b23d-0822be5fd260',
 '72c664df-5537-4e8c-a4c5-cf0e16edb39b',
 '61c03da3-6cfb-4a5e-a6f5-f45f0c02df1e',
 '189ed4a2-557c-454f-a819-52a689ba2075',
 '47bb8a34-c345-4699-af7f-e7b57b23eda5',
 '0a5c63fa-563e-43b9-8f23-e2fdcd002328',
 '65ee8555-7c11-4d07-aa93-f221511b8dc6',
 'f4c6eedd-e34e-4cc1-a63c-f81e86bc7220',
 'a75415ab-916a-4aee-b3d7-19609c807cc8',
 '88f42358-b240-471b-8953-e99524d46ddb',
 'bfa70579-ab7b-486e-88de-730b579b0805',
 '888c0a0c-b606-4a27-99ef-f5331a814a2c',
 'bca30df8-03a4-45ef-92ae-b946e7688e27',
 'eac0b30d-54e0-4c4a-9d8b-e1bfc82bfa32',
 '26b5c621-8330-4661-a69e-491d8c2687bd',
 'a84c100e-d291-4a08-b777-0bf40ab59fff',
 'abe6c59c-e7ea-447a-a103-81d107b197f8',
 '2a33f48c-4646-4235-9973-da276d4e2572',
 'd4b4b97e-0d22-40ba-acd6-0d814e26140c',
 '1416a504-0633-4c92-98a0-1c9c4bb26ca4',
 '8a299e1a-848e-

In [33]:
retriever = hypothetical_questions_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={'k': 5}
)

In [34]:
user_input = "Any specific fines levied on the company in 2022?"

In [37]:
hypothetical_questions_retrieved = retriever.invoke(user_input)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [38]:
hypothetical_questions_retrieved

[Document(id='83649b38-8013-4d76-b231-505cbe279e8e', metadata={'parent_chunk_id': 'text_3300', 'parent_collection': 'tesla-10k-2019-to-2023'}, page_content=''),
 Document(id='c132caaa-eaae-41cc-a029-9cbfbb32da65', metadata={'parent_chunk_id': 'text_3322', 'parent_collection': 'tesla-10k-2019-to-2023'}, page_content=''),
 Document(id='ffc6fdb1-615a-4527-bddb-212d293221a5', metadata={'parent_chunk_id': 'text_3323', 'parent_collection': 'tesla-10k-2019-to-2023'}, page_content=''),
 Document(id='6e0d6152-55b9-43e2-aa89-09d61d3410de', metadata={'parent_chunk_id': 'text_3324', 'parent_collection': 'tesla-10k-2019-to-2023'}, page_content=''),
 Document(id='be39a3cc-e03d-4053-8e5b-00d5d7a25599', metadata={'parent_chunk_id': 'text_3326', 'parent_collection': 'tesla-10k-2019-to-2023'}, page_content='')]

In [ ]:
retrived_documents = vectorstore.get(
    ids=list(set([d.metadata['parent_chunk_id'] for d in hypothetical_questions_retrieved]))
)

In [ ]:
print(retrived_documents)

In [ ]:
retrived_documents['documents']

In [ ]:
len(retrived_documents['documents'])

In [ ]:
qna_system_message = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

User queries will be delimited by: <Question> and </Question>.

Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

If the answer is not found in the context, respond "I don't know".
"""

In [ ]:
qna_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""

In [ ]:
user_input = "Any specific fines levied on the company in 2022?"

In [ ]:
model_name = 'openai/gpt-oss-120b'

prompt = [
    {'role': 'system', 'content': qna_system_message},
    {'role': 'user', 'content': qna_user_message_template.format(
         context=retrived_documents,
         question=user_input
        )
    }
]

try:
    response = client.chat.completions.create(
        model=model_name,
        messages=prompt,
        temperature=0
    )

    prediction = response.choices[0].message.content.strip()
except Exception as e:
    prediction = f'Sorry, I encountered the following error: \n {e}'

print(prediction)
